# Clustering with K-means and DBSCAN

Explore compact clusters, curved clusters, density-based noise, inertia, and silhouette score. These are generated datasets with known labels for illustration; the labels are used only after clustering for comparison.

Clustering this complete exploratory sample is not a held-out prediction benchmark. No files or GPU are needed.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

SEED = 42
blobs, blob_truth = make_blobs(n_samples=350, centers=3, cluster_std=0.8, random_state=SEED)
blob_features = StandardScaler().fit_transform(blobs)


## 1. Inspect a small range of K-means cluster counts

Inertia decreases as more clusters are allowed; its minimum alone cannot choose a useful `k`. Silhouette compares within-cluster cohesion with separation. It also has limitations for non-convex shapes.


In [ ]:
rows = []
for k in range(2, 7):
    clusterer = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(blob_features)
    rows.append({'k': k, 'inertia': clusterer.inertia_,
                 'silhouette': silhouette_score(blob_features, clusterer.labels_)})
results = pd.DataFrame(rows)
display(results.round(3))
chosen_k = int(results.loc[results['silhouette'].idxmax(), 'k'])
blob_labels = KMeans(n_clusters=chosen_k, n_init=10, random_state=SEED).fit_predict(blob_features)
print('Exploratory selected k:', chosen_k)
print('Agreement with synthetic labels:', round(adjusted_rand_score(blob_truth, blob_labels), 3))


## 2. Compare shapes using the same scaled features

DBSCAN identifies points with label `-1` as noise. Its `eps` depends on feature scale. There is no general rule that makes one clustering method best for every dataset.


In [ ]:
moons, moon_truth = make_moons(n_samples=400, noise=0.06, random_state=SEED)
moon_features = StandardScaler().fit_transform(moons)
km_labels = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(moon_features)
db_labels = DBSCAN(eps=0.22, min_samples=5).fit_predict(moon_features)
cluster_count = len(set(db_labels) - {-1})
print('DBSCAN clusters:', cluster_count, 'Noise points:', int((db_labels == -1).sum()))
print('K-means label agreement:', round(adjusted_rand_score(moon_truth, km_labels), 3))
print('DBSCAN label agreement:', round(adjusted_rand_score(moon_truth, db_labels), 3))
usable = db_labels != -1
if 1 < len(np.unique(db_labels[usable])) < usable.sum():
    print('DBSCAN silhouette excluding noise:', round(silhouette_score(moon_features[usable], db_labels[usable]), 3))

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, labels, title in zip(axes, [moon_truth, km_labels, db_labels],
                           ['Synthetic labels', 'K-means', 'DBSCAN (-1 is noise)']):
    ax.scatter(moon_features[:, 0], moon_features[:, 1], c=labels, s=12, cmap='tab10')
    ax.set_title(title)
fig.tight_layout()
plt.show()


## Practice

Change noise or density, then explain failures rather than choosing settings from the synthetic labels alone. DBSCAN has no standard out-of-sample `predict` method; do not treat its fitted labels as a deployed classifier.

**Reference:** [scikit-learn: clustering](https://scikit-learn.org/stable/modules/clustering.html).
